In [1]:
import json

# json 파일 하나 열어서 구조 확인
json_path = "./datasets/COCO_motorcycle (pixel).json"  # 실제 경로로 변경
with open(json_path, "r") as f:
    data = json.load(f)

print(type(data))
if isinstance(data, dict):
    print("키 목록:", data.keys())
elif isinstance(data, list):
    print("리스트 길이:", len(data))
    print("첫번째 항목 키:", data[0].keys())

<class 'dict'>
키 목록: dict_keys(['info', 'licenses', 'images', 'annotations', 'categories'])


In [2]:
# 각 키별 내용 확인
print("=== info ===")
print(data['info'])

print("\n=== categories ===")
print(data['categories'])

print("\n=== images 첫번째 ===")
print(data['images'][0])

print("\n=== annotations 첫번째 ===")
print(data['annotations'][0])

print("\n=== 전체 통계 ===")
print(f"이미지 수: {len(data['images'])}")
print(f"어노테이션 수: {len(data['annotations'])}")
print(f"카테고리 수: {len(data['categories'])}")

=== info ===
{'description': 'This is dataset.', 'url': 'https://superannotate.ai', 'version': '1.0', 'year': 2022, 'contributor': 'Superannotate AI', 'date_created': '15/09/2022'}

=== categories ===
[{'id': 1329681, 'name': 'Rider', 'supercategory': 'Rider', 'isthing': 1, 'color': [17, 74, 20]}, {'id': 1323885, 'name': 'My bike', 'supercategory': 'My bike', 'isthing': 1, 'color': [109, 51, 20]}, {'id': 1323884, 'name': 'Moveable', 'supercategory': 'Moveable', 'isthing': 1, 'color': [108, 51, 20]}, {'id': 1323882, 'name': 'Lane Mark', 'supercategory': 'Lane Mark', 'isthing': 1, 'color': [106, 51, 20]}, {'id': 1323881, 'name': 'Road', 'supercategory': 'Road', 'isthing': 1, 'color': [105, 51, 20]}, {'id': 1323880, 'name': 'Undrivable', 'supercategory': 'Undrivable', 'isthing': 1, 'color': [104, 51, 20]}]

=== images 첫번째 ===
{'id': 1, 'file_name': 'night ride (8).png', 'height': 1080, 'width': 1920, 'license': 1}

=== annotations 첫번째 ===
{'id': 1, 'image_id': 1, 'segmentation': [[0, 0, 0

# 1. json파일 yolo형태로 데이터 준비

In [1]:
import json
import os
from collections import defaultdict
from tqdm import tqdm

with open("./datasets/COCO_motorcycle (pixel).json", "r") as f:
    print("JSON 로딩 중...")
    data = json.load(f)
    print(f"완료: 이미지 {len(data['images'])}개, 어노테이션 {len(data['annotations'])}개")

cat_id_to_yolo = {
    #1323880: 0,  # Undrivable
    #1323881: 1,  # Road
    1323882: 0,  # Lane Mark
    1323884: 1,  # Moveable
    1323885: 2,  # My bike
    1329681: 3,  # Rider
}

# 여기서 학습에서 제외할 클래스 지정 (YOLO class index 기준)
EXCLUDE_CLASSES = {2, 3}  # Undrivable, Road 제외
# EXCLUDE_CLASSES = {}  # 전부 포함할 때는 빈 set

image_info = {img["id"]: img for img in data["images"]}

ann_by_image = defaultdict(list)
for ann in data["annotations"]:
    ann_by_image[ann["image_id"]].append(ann)

output_dir = "./datasets/yolo/labels"
os.makedirs(output_dir, exist_ok=True)

for image_id, img in tqdm(image_info.items(), total=len(image_info), desc="변환 중"):
    file_name = img["file_name"]
    img_w = img["width"]
    img_h = img["height"]
    
    txt_name = os.path.splitext(file_name)[0] + ".txt"
    txt_path = os.path.join(output_dir, txt_name)
    
    anns = ann_by_image[image_id]
    
    with open(txt_path, "w") as f:
        for ann in anns:
            cls = cat_id_to_yolo.get(ann["category_id"])
            if cls is None:
                continue
            
            # 제외 클래스 스킵
            if cls in EXCLUDE_CLASSES:
                continue
            
            for seg in ann["segmentation"]:
                coords = []
                for i in range(0, len(seg), 2):
                    x = max(0, min(1, seg[i] / img_w))
                    y = max(0, min(1, seg[i+1] / img_h))
                    coords.append(f"{x:.5f} {y:.5f}")
                
                f.write(f"{cls} {' '.join(coords)}\n")

print("변환 완료!")

JSON 로딩 중...
완료: 이미지 200개, 어노테이션 2305개


변환 중: 100%|██████████| 200/200 [00:01<00:00, 164.86it/s]

변환 완료!


# 2. 전이학습

In [5]:
# 2. 전이학습
from ultralytics import YOLO

model = YOLO("yolov8n-seg.pt")

results = model.train(
    data="./datasets/yolo/data.yaml",
    epochs=50,
    imgsz=640,
    batch=8,
    lr0=0.001,
    patience=10,
    device=0,
    workers=4,
    project="./runs",
    name="two_class_finetune",
    pretrained=True,
    freeze=10,
)

print(f"모델 특화 모델: {results.save_dir}/weights/best.pt")

Ultralytics 8.4.41  Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24563MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./datasets/yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=two_class_finetune-3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import time
from collections import deque
import threading
import queue

model = YOLO("./runs/segment/runs/two_class_finetune-3/weights/best.pt")

cap = cv2.VideoCapture("./datasets/video_480p.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

output_path = "./FT_2_classes_480p_enhanced_mt_super_fast.mp4"
out = cv2.VideoWriter(
    output_path,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps, (width, height)
)

COLORS = {
    0: (0, 255, 255),
    1: (0, 0, 125),
    2: (0, 100, 0),
    3: (0, 0, 255),
    4: (0, 255, 0),
}

EDGE_MARGIN = 20
FRAME_AREA = width * height
NEARBY_AREA_RATIO = 0.015
NEARBY_BOTTOM_RATIO = 0.75
PEDESTRIAN_RATIO = 1.8

kernel_small = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))

# 스레드간 데이터 전달용 큐
frame_queue = queue.Queue(maxsize=16)    # 원본 프레임
result_queue = queue.Queue(maxsize=16)   # 처리된 프레임
write_queue = queue.Queue(maxsize=16)    # 저장할 프레임

done_flag = threading.Event()
fps_deque = deque(maxlen=30)

def is_nearby(x1, y1, x2, y2):
    box_area = (x2 - x1) * (y2 - y1)
    area_ratio = box_area / FRAME_AREA
    bottom_ratio = y2 / height
    if area_ratio >= NEARBY_AREA_RATIO and bottom_ratio >= NEARBY_BOTTOM_RATIO:
        return True
    if area_ratio >= NEARBY_AREA_RATIO * 3:
        return True
    return False

def get_final_class(cls, x1, y1, x2, y2):
    if cls == 0:
        return 0
    elif cls == 1:
        box_w = x2 - x1
        box_h = y2 - y1
        ratio = box_h / box_w if box_w > 0 else 999
        nearby = is_nearby(x1, y1, x2, y2)
        if ratio >= PEDESTRIAN_RATIO:
            if x1 <= EDGE_MARGIN or x2 >= width - EDGE_MARGIN:
                return 3 if nearby else 1
            else:
                return 4
        else:
            return 3 if nearby else 1
    return cls

def process_mask(mask):
    mask_blur = cv2.GaussianBlur(mask, (5, 5), 0)
    _, mask_bin = cv2.threshold(mask_blur, 0.5, 1, cv2.THRESH_BINARY)
    mask_clean = cv2.morphologyEx(mask_bin, cv2.MORPH_CLOSE, kernel_small)
    mask_clean = cv2.morphologyEx(mask_clean, cv2.MORPH_OPEN, kernel_small)
    mask_resized = cv2.resize(mask_clean, (width, height), interpolation=cv2.INTER_LINEAR)
    return mask_resized > 0.5

# ── 스레드 1: 프레임 읽기 ──────────────────────────
def read_frames():
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            frame_queue.put(None)  # 종료 신호
            break
        frame_queue.put(frame)
    print("[Reader] 완료")

# ── 스레드 2: YOLO 추론 + 마스크 처리 ──────────────
def inference_worker():
    processed = 0
    while True:
        frame = frame_queue.get()
        if frame is None:
            write_queue.put(None)  # 종료 신호 전달
            break

        t_start = time.time()

        results = model(frame, conf=0.4, verbose=False)[0]

        if results.masks is not None:
            masks = results.masks.data.cpu().numpy()
            boxes = results.boxes.xyxy.cpu().numpy().astype(int)
            classes = results.boxes.cls.cpu().numpy().astype(int)

            final_classes = []
            for cls, box in zip(classes, boxes):
                x1, y1, x2, y2 = box
                final_classes.append(get_final_class(cls, x1, y1, x2, y2))

            overlay = frame.copy()
            for mask, cls in zip(masks, final_classes):
                if cls == -1:
                    continue
                
                # 1 마스크 블러처리
                # mask_bool = process_mask(mask)
                # color = COLORS.get(cls, (255, 255, 255))
                # overlay[mask_bool] = color
                
                # 2 블러 없이 바로 사용
                overlay[mask.astype(bool)] = color  # 블러 없이 바로 사용

            frame = cv2.addWeighted(frame, 0.6, overlay, 0.4, 0)

            for box, cls in zip(boxes, final_classes):
                if cls not in (2, 4):
                    continue
                x1, y1, x2, y2 = box
                color = COLORS.get(cls, (255, 255, 255))
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, "Pedestrian",
                           (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        t_end = time.time()
        fps_deque.append(1.0 / (t_end - t_start))
        processed += 1

        if processed % 50 == 0:
            avg_fps = sum(fps_deque) / len(fps_deque)
            remaining = (total_frames - processed) / avg_fps
            print(f"진행: {processed}/{total_frames} | FPS: {avg_fps:.1f} | 남은시간: {remaining:.0f}초")

        write_queue.put(frame)

    print("[Inference] 완료")

# ── 스레드 3: 파일 저장 ────────────────────────────
def write_frames():
    while True:
        frame = write_queue.get()
        if frame is None:
            break
        out.write(frame)
    print("[Writer] 완료")

# 스레드 시작
t1 = threading.Thread(target=read_frames, daemon=True)
t2 = threading.Thread(target=inference_worker, daemon=True)
t3 = threading.Thread(target=write_frames, daemon=True)

print(f"총 {total_frames}프레임 처리 시작 (멀티스레딩)")
t_total_start = time.time()

t1.start()
t2.start()
t3.start()

t1.join()
t2.join()
t3.join()

cap.release()
out.release()

elapsed = time.time() - t_total_start
avg_fps = sum(fps_deque) / len(fps_deque) if fps_deque else 0
print(f"\n완료: {output_path}")
print(f"총 소요시간: {elapsed:.1f}초")
print(f"평균 FPS: {avg_fps:.1f}")

총 3624프레임 처리 시작 (멀티스레딩)
진행: 50/3624 | FPS: 39.7 | 남은시간: 90초
진행: 100/3624 | FPS: 46.5 | 남은시간: 76초
진행: 150/3624 | FPS: 53.8 | 남은시간: 65초
진행: 200/3624 | FPS: 55.0 | 남은시간: 62초
진행: 250/3624 | FPS: 51.4 | 남은시간: 66초
진행: 300/3624 | FPS: 48.7 | 남은시간: 68초
진행: 350/3624 | FPS: 43.8 | 남은시간: 75초
진행: 400/3624 | FPS: 52.9 | 남은시간: 61초
진행: 450/3624 | FPS: 47.5 | 남은시간: 67초
진행: 500/3624 | FPS: 61.2 | 남은시간: 51초
진행: 550/3624 | FPS: 44.7 | 남은시간: 69초
진행: 600/3624 | FPS: 55.8 | 남은시간: 54초
진행: 650/3624 | FPS: 49.5 | 남은시간: 60초
진행: 700/3624 | FPS: 52.2 | 남은시간: 56초
진행: 750/3624 | FPS: 59.4 | 남은시간: 48초
진행: 800/3624 | FPS: 59.6 | 남은시간: 47초
진행: 850/3624 | FPS: 44.1 | 남은시간: 63초
진행: 900/3624 | FPS: 53.7 | 남은시간: 51초
진행: 950/3624 | FPS: 60.1 | 남은시간: 44초
진행: 1000/3624 | FPS: 72.7 | 남은시간: 36초
진행: 1050/3624 | FPS: 34.3 | 남은시간: 75초
진행: 1100/3624 | FPS: 33.5 | 남은시간: 75초
진행: 1150/3624 | FPS: 48.5 | 남은시간: 51초
진행: 1200/3624 | FPS: 51.3 | 남은시간: 47초
진행: 1250/3624 | FPS: 68.0 | 남은시간: 35초
진행: 1300/3624 | FPS: 86.2 | 남은시간: 27초
진행: 1350

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import time
from collections import deque
import threading
import queue

model = YOLO("./runs/segment/runs/two_class_finetune-3/weights/best.pt")

cap = cv2.VideoCapture("./datasets/video_480p.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

output_path = "./FT_2_classes_480p_enhanced_mt.mp4"
out = cv2.VideoWriter(
    output_path,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps, (width, height)
)

COLORS = {
    0: (0, 255, 255),
    1: (0, 0, 80),
    2: (0, 100, 0),
    3: (0, 0, 255),
    4: (0, 255, 0),
}

EDGE_MARGIN = 20
FRAME_AREA = width * height
NEARBY_AREA_RATIO = 0.015
NEARBY_BOTTOM_RATIO = 0.75
PEDESTRIAN_RATIO = 1.8

kernel_small = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))

raw_queue = queue.Queue(maxsize=16)      # 원본 프레임
infer_queue = queue.Queue(maxsize=16)    # 추론 결과
postproc_queue = queue.Queue(maxsize=16) # 후처리 결과
write_queue = queue.Queue(maxsize=16)    # 최종 프레임

done_flag = threading.Event()
fps_deque = deque(maxlen=30)

def is_nearby(x1, y1, x2, y2):
    box_area = (x2 - x1) * (y2 - y1)
    area_ratio = box_area / FRAME_AREA
    bottom_ratio = y2 / height
    if area_ratio >= NEARBY_AREA_RATIO and bottom_ratio >= NEARBY_BOTTOM_RATIO:
        return True
    if area_ratio >= NEARBY_AREA_RATIO * 3:
        return True
    return False

def get_final_class(cls, x1, y1, x2, y2):
    if cls == 0:
        return 0
    elif cls == 1:
        box_w = x2 - x1
        box_h = y2 - y1
        ratio = box_h / box_w if box_w > 0 else 999
        nearby = is_nearby(x1, y1, x2, y2)
        if ratio >= PEDESTRIAN_RATIO:
            if x1 <= EDGE_MARGIN or x2 >= width - EDGE_MARGIN:
                return 3 if nearby else 1
            else:
                return 4
        else:
            return 3 if nearby else 1
    return cls

def process_mask(mask):
    mask_blur = cv2.GaussianBlur(mask, (5, 5), 0)
    _, mask_bin = cv2.threshold(mask_blur, 0.5, 1, cv2.THRESH_BINARY)
    mask_clean = cv2.morphologyEx(mask_bin, cv2.MORPH_CLOSE, kernel_small)
    mask_clean = cv2.morphologyEx(mask_clean, cv2.MORPH_OPEN, kernel_small)
    mask_resized = cv2.resize(mask_clean, (width, height), interpolation=cv2.INTER_LINEAR)
    return mask_resized > 0.5

# ── 스레드 1: 프레임 읽기 ──────────────────────────
def read_frames():
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            raw_queue.put(None)
            break
        raw_queue.put(frame)
        
# 스레드 2: YOLO 추론만 (GPU)
def inference_worker():
    while True:
        frame = raw_queue.get()
        if frame is None:
            infer_queue.put(None)
            break
        results = model(frame, conf=0.4, verbose=False)[0]
        infer_queue.put((frame, results))

# 스레드 3: 마스크 후처리 (CPU - 13900K 활용)
def postprocess_worker():
    processed = 0
    while True:
        item = infer_queue.get()
        if item is None:
            postproc_queue.put(None)
            break

        t_start = time.time()
        frame, results = item

        if results.masks is not None:
            masks = results.masks.data.cpu().numpy()
            boxes = results.boxes.xyxy.cpu().numpy().astype(int)
            classes = results.boxes.cls.cpu().numpy().astype(int)

            final_classes = []
            for cls, box in zip(classes, boxes):
                x1, y1, x2, y2 = box
                final_classes.append(get_final_class(cls, x1, y1, x2, y2))

            overlay = frame.copy()
            for mask, cls in zip(masks, final_classes):
                if cls == -1:
                    continue
                
                color = COLORS.get(cls, (255, 255, 255))  # color 반드시 여기서 정의
        
                # 1 마스크 블러처리
                # mask_bool = process_mask(mask)
                # overlay[mask_bool] = color
                
                # 2 블러 없이 바로 사용 (160x160 → 원본 해상도로 리사이즈 필수)
                mask_resized = cv2.resize(mask, (width, height)) > 0.5
                overlay[mask_resized] = color

            frame = cv2.addWeighted(frame, 0.6, overlay, 0.4, 0)

            for box, cls in zip(boxes, final_classes):
                if cls not in (2, 3, 4):
                    continue
                x1, y1, x2, y2 = box
                color = COLORS.get(cls, (255, 255, 255))
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, "Pedestrian",
                           (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        t_end = time.time()
        elapsed = t_end - t_start
        
        # 0으로 나누기 방지
        if elapsed > 0:
            fps_deque.append(1.0 / elapsed)
        
        processed += 1

        if processed % 50 == 0:
            # fps_deque 비어있을 때 방지
            if len(fps_deque) > 0:
                avg_fps = sum(fps_deque) / len(fps_deque)
                remaining = (total_frames - processed) / avg_fps if avg_fps > 0 else 0
                print(f"진행: {processed}/{total_frames} | FPS: {avg_fps:.1f} | 남은시간: {remaining:.0f}초")

        postproc_queue.put(frame)

    print("[PostProcess] 완료")

# 스레드 4: 화면출력 + 파일저장
def write_frames():
    while True:
        frame = postproc_queue.get()
        if frame is None:
            break
        avg_fps = sum(fps_deque) / len(fps_deque) if fps_deque else 0
        cv2.putText(frame, f"FPS: {avg_fps:.1f}", (10, 30),
                   cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
        out.write(frame)
        cv2.imshow("Segmentation", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    cv2.destroyAllWindows()
    print("[Writer] 완료")

# 스레드 시작
t1 = threading.Thread(target=read_frames, daemon=True)
t2 = threading.Thread(target=inference_worker, daemon=True)
t3 = threading.Thread(target=postprocess_worker, daemon=True)
t4 = threading.Thread(target=write_frames, daemon=True)

print(f"총 {total_frames}프레임 처리 시작 (멀티스레딩)")
t_total_start = time.time()

t1.start()
t2.start()
t3.start()
t4.start()

t1.join()
t2.join()
t3.join()
t4.join()

cap.release()
out.release()

elapsed = time.time() - t_total_start
avg_fps = sum(fps_deque) / len(fps_deque) if fps_deque else 0
print(f"\n완료: {output_path}")
print(f"총 소요시간: {elapsed:.1f}초")
print(f"평균 FPS: {avg_fps:.1f}")

총 3624프레임 처리 시작 (멀티스레딩)
진행: 50/3624 | FPS: 54.4 | 남은시간: 66초
진행: 100/3624 | FPS: 63.6 | 남은시간: 55초
진행: 150/3624 | FPS: 116.8 | 남은시간: 30초
진행: 200/3624 | FPS: 119.2 | 남은시간: 29초
진행: 250/3624 | FPS: 148.7 | 남은시간: 23초
진행: 300/3624 | FPS: 122.0 | 남은시간: 27초
진행: 350/3624 | FPS: 94.8 | 남은시간: 35초
진행: 400/3624 | FPS: 110.6 | 남은시간: 29초
진행: 450/3624 | FPS: 98.9 | 남은시간: 32초
진행: 500/3624 | FPS: 135.7 | 남은시간: 23초
진행: 550/3624 | FPS: 94.1 | 남은시간: 33초
진행: 600/3624 | FPS: 145.7 | 남은시간: 21초
진행: 650/3624 | FPS: 105.5 | 남은시간: 28초
진행: 700/3624 | FPS: 136.7 | 남은시간: 21초
진행: 750/3624 | FPS: 138.0 | 남은시간: 21초
진행: 800/3624 | FPS: 128.7 | 남은시간: 22초
진행: 850/3624 | FPS: 90.8 | 남은시간: 31초
진행: 900/3624 | FPS: 109.6 | 남은시간: 25초
진행: 950/3624 | FPS: 130.4 | 남은시간: 20초
진행: 1000/3624 | FPS: 253.9 | 남은시간: 10초
진행: 1050/3624 | FPS: 71.6 | 남은시간: 36초
진행: 1100/3624 | FPS: 66.7 | 남은시간: 38초
진행: 1150/3624 | FPS: 103.0 | 남은시간: 24초
진행: 1200/3624 | FPS: 133.2 | 남은시간: 18초
진행: 1250/3624 | FPS: 161.2 | 남은시간: 15초
진행: 1300/3624 | FPS: 168.6 | 

2분 48초

# 마스크 병목 발생부분 최적화
변경 내용 요약:
항목기존변경 후마스크 리사이즈마스크 N개 × cv2.resize 호출전체를 옆으로 이어붙여 1회 resize컬러 적용마스크별 루프 + 픽셀 인덱싱class_canvas → LUT 벡터 인덱싱 (루프 없음)블렌딩cv2.addWeighted (전체 프레임)마스크 영역만 numpy 슬라이스 블렌딩overlay.copy()항상 전체 복사탐지 없으면 복사 생략
퀄리티 변화: 없음 — 연산 결과가 수학적으로 동일합니다. 블렌딩 계수 0.6/0.4도 그대로이고, 리사이즈 방법(INTER_LINEAR)도 동일합니다.
예상 속도: 마스크 수가 많을수록 (5개 이상) 효과가 큽니다. cv2.resize 반복 호출 오버헤드가 제거되고 LUT 인덱싱은 루프 대비 수십 배 빠릅니다.

In [1]:
from ultralytics import YOLO
import cv2
import numpy as np
import time
from collections import deque
import threading
import queue

model = YOLO("./runs/segment/runs/two_class_finetune-3/weights/best.pt")

cap = cv2.VideoCapture("./datasets/video_480p.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

output_path = "./FT_2_classes_480p_enhanced_mt.mp4"
out = cv2.VideoWriter(
    output_path,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps, (width, height)
)

COLORS = {
    0: (0, 255, 255),
    1: (0, 0, 80),
    2: (0, 100, 0),
    3: (0, 0, 255),
    4: (0, 255, 0),
}

EDGE_MARGIN = 20
FRAME_AREA = width * height
NEARBY_AREA_RATIO = 0.015
NEARBY_BOTTOM_RATIO = 0.75
PEDESTRIAN_RATIO = 1.8

kernel_small = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))

raw_queue = queue.Queue(maxsize=16)      # 원본 프레임
infer_queue = queue.Queue(maxsize=16)    # 추론 결과
postproc_queue = queue.Queue(maxsize=16) # 후처리 결과
write_queue = queue.Queue(maxsize=16)    # 최종 프레임

done_flag = threading.Event()
fps_deque = deque(maxlen=30)

def is_nearby(x1, y1, x2, y2):
    box_area = (x2 - x1) * (y2 - y1)
    area_ratio = box_area / FRAME_AREA
    bottom_ratio = y2 / height
    if area_ratio >= NEARBY_AREA_RATIO and bottom_ratio >= NEARBY_BOTTOM_RATIO:
        return True
    if area_ratio >= NEARBY_AREA_RATIO * 3:
        return True
    return False

def get_final_class(cls, x1, y1, x2, y2):
    if cls == 0:
        return 0
    elif cls == 1:
        box_w = x2 - x1
        box_h = y2 - y1
        ratio = box_h / box_w if box_w > 0 else 999
        nearby = is_nearby(x1, y1, x2, y2)
        if ratio >= PEDESTRIAN_RATIO:
            if x1 <= EDGE_MARGIN or x2 >= width - EDGE_MARGIN:
                return 3 if nearby else 1
            else:
                return 4
        else:
            return 3 if nearby else 1
    return cls

def process_mask(mask):
    mask_blur = cv2.GaussianBlur(mask, (5, 5), 0)
    _, mask_bin = cv2.threshold(mask_blur, 0.5, 1, cv2.THRESH_BINARY)
    mask_clean = cv2.morphologyEx(mask_bin, cv2.MORPH_CLOSE, kernel_small)
    mask_clean = cv2.morphologyEx(mask_clean, cv2.MORPH_OPEN, kernel_small)
    mask_resized = cv2.resize(mask_clean, (width, height), interpolation=cv2.INTER_LINEAR)
    return mask_resized > 0.5

# ── 스레드 1: 프레임 읽기 ──────────────────────────
def read_frames():
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            raw_queue.put(None)
            break
        raw_queue.put(frame)
        
# 스레드 2: YOLO 추론만 (GPU)
def inference_worker():
    while True:
        frame = raw_queue.get()
        if frame is None:
            infer_queue.put(None)
            break
        results = model(frame, conf=0.4, verbose=False)[0]
        infer_queue.put((frame, results))
        
# 스레드 3: 마스크 후처리 (CPU - 13900K 활용)
def postprocess_worker():
    # 컬러 LUT: 클래스 인덱스 → BGR 배열 (최대 클래스 수만큼)
    COLOR_LUT = np.zeros((10, 3), dtype=np.uint8)
    for cls_id, bgr in COLORS.items():
        COLOR_LUT[cls_id] = bgr

    processed = 0
    while True:
        item = infer_queue.get()
        if item is None:
            postproc_queue.put(None)
            break

        t_start = time.time()
        frame, results = item

        if results.masks is not None:
            masks = results.masks.data.cpu().numpy()   # (N, mH, mW) float32
            boxes = results.boxes.xyxy.cpu().numpy().astype(int)
            classes = results.boxes.cls.cpu().numpy().astype(int)

            final_classes = [get_final_class(cls, *box) for cls, box in zip(classes, boxes)]
            n = len(final_classes)

            # ── 핵심 최적화: 마스크 N장을 한 번에 리사이즈 ──────────────
            # (N, mH, mW) → (N, height, width) 배치 리사이즈
            # cv2.resize는 배치 미지원 → numpy로 한 번에 처리
            mH, mW = masks.shape[1], masks.shape[2]
            if mH != height or mW != width:
                # (N, mH, mW) → (mH, mW*N) 로 옆으로 이어붙여 한 번만 resize
                masks_concat = masks.transpose(1, 0, 2).reshape(mH, n * mW)  # (mH, N*mW)
                masks_resized_concat = cv2.resize(
                    masks_concat, (n * width, height), interpolation=cv2.INTER_LINEAR
                )
                # 다시 (height, N, width) → (N, height, width)
                masks_resized = masks_resized_concat.reshape(height, n, width).transpose(1, 0, 2)
            else:
                masks_resized = masks  # 이미 원본 해상도인 경우

            # ── 컬러 오버레이를 단일 배열로 합성 ──────────────────────────
            # class_canvas: 각 픽셀에 해당 클래스 ID 저장 (나중에 온 마스크가 위에 덮음)
            # -1 = 배경
            class_canvas = np.full((height, width), -1, dtype=np.int8)
            for i, cls in enumerate(final_classes):
                if cls == -1:
                    continue
                mask_bool = masks_resized[i] > 0.5
                class_canvas[mask_bool] = cls

            # class_canvas → BGR 컬러 오버레이 (LUT 인덱싱, 루프 없음)
            has_detection = class_canvas >= 0
            overlay = frame.copy()
            valid_cls = class_canvas[has_detection].astype(np.int64)
            overlay[has_detection] = COLOR_LUT[valid_cls]

            # addWeighted 대신 슬라이스 블렌딩 (has_detection 영역만)
            # 퀄리티 동일, 전체 프레임 연산 대신 마스크 영역만
            alpha = 0.4
            frame_out = frame.copy()
            frame_out[has_detection] = (
                frame[has_detection] * 0.6 + overlay[has_detection] * alpha
            ).astype(np.uint8)

            # ── 바운딩박스 + 텍스트 (cls 2, 4만) ─────────────────────────
            for box, cls in zip(boxes, final_classes):
                if cls not in (2, 4):
                    continue
                x1, y1, x2, y2 = box
                color = COLORS.get(cls, (255, 255, 255))
                cv2.rectangle(frame_out, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame_out, "Pedestrian",
                            (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        else:
            frame_out = frame

        t_end = time.time()
        elapsed = t_end - t_start
        if elapsed > 0:
            fps_deque.append(1.0 / elapsed)

        processed += 1
        if processed % 50 == 0:
            if len(fps_deque) > 0:
                avg_fps = sum(fps_deque) / len(fps_deque)
                remaining = (total_frames - processed) / avg_fps if avg_fps > 0 else 0
                print(f"진행: {processed}/{total_frames} | FPS: {avg_fps:.1f} | 남은시간: {remaining:.0f}초")

        postproc_queue.put(frame_out)

    print("[PostProcess] 완료")        


# 스레드 4: 화면출력 + 파일저장
def write_frames():
    while True:
        frame = postproc_queue.get()
        if frame is None:
            break
        avg_fps = sum(fps_deque) / len(fps_deque) if fps_deque else 0
        cv2.putText(frame, f"FPS: {avg_fps:.1f}", (10, 30),
                   cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
        out.write(frame)
        cv2.imshow("Segmentation", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    cv2.destroyAllWindows()
    print("[Writer] 완료")

# 스레드 시작
t1 = threading.Thread(target=read_frames, daemon=True)
t2 = threading.Thread(target=inference_worker, daemon=True)
t3 = threading.Thread(target=postprocess_worker, daemon=True)
t4 = threading.Thread(target=write_frames, daemon=True)

print(f"총 {total_frames}프레임 처리 시작 (멀티스레딩)")
t_total_start = time.time()

t1.start()
t2.start()
t3.start()
t4.start()

t1.join()
t2.join()
t3.join()
t4.join()

cap.release()
out.release()

elapsed = time.time() - t_total_start
avg_fps = sum(fps_deque) / len(fps_deque) if fps_deque else 0
print(f"\n완료: {output_path}")
print(f"총 소요시간: {elapsed:.1f}초")
print(f"평균 FPS: {avg_fps:.1f}")

총 3624프레임 처리 시작 (멀티스레딩)
진행: 50/3624 | FPS: 83.5 | 남은시간: 43초
진행: 100/3624 | FPS: 95.6 | 남은시간: 37초
진행: 150/3624 | FPS: 103.6 | 남은시간: 34초
진행: 200/3624 | FPS: 119.7 | 남은시간: 29초
진행: 250/3624 | FPS: 120.4 | 남은시간: 28초
진행: 300/3624 | FPS: 113.6 | 남은시간: 29초
진행: 350/3624 | FPS: 93.7 | 남은시간: 35초
진행: 400/3624 | FPS: 111.2 | 남은시간: 29초
진행: 450/3624 | FPS: 97.7 | 남은시간: 32초
진행: 500/3624 | FPS: 136.1 | 남은시간: 23초
진행: 550/3624 | FPS: 104.5 | 남은시간: 29초
진행: 600/3624 | FPS: 130.2 | 남은시간: 23초
진행: 650/3624 | FPS: 96.5 | 남은시간: 31초
진행: 700/3624 | FPS: 120.5 | 남은시간: 24초
진행: 750/3624 | FPS: 108.5 | 남은시간: 26초
진행: 800/3624 | FPS: 103.0 | 남은시간: 27초
진행: 850/3624 | FPS: 87.2 | 남은시간: 32초
진행: 900/3624 | FPS: 105.2 | 남은시간: 26초
진행: 950/3624 | FPS: 112.9 | 남은시간: 24초
진행: 1000/3624 | FPS: 163.3 | 남은시간: 16초
진행: 1050/3624 | FPS: 76.2 | 남은시간: 34초
진행: 1100/3624 | FPS: 85.4 | 남은시간: 30초
진행: 1150/3624 | FPS: 93.1 | 남은시간: 27초
진행: 1200/3624 | FPS: 114.9 | 남은시간: 21초
진행: 1250/3624 | FPS: 98.8 | 남은시간: 24초
진행: 1300/3624 | FPS: 136.7 | 남은